# Leidenfrost Testcase
This is a testcase for sliding Leidenforst droplet on soft solid. 

`https://www.doi.org/10.1140/epje/s10189-026-00606-7`

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
//using BoSSS.Application.XNSFE_Solver;
using HFSISolver;
using HFSISolver.SolidPhase;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [3]:
string proj = "HFSI-Leidenfrost-Angle";

In [ ]:
static var myBatch = BoSSSshell.GetDefaultQueue();

In [5]:
BoSSSshell.WorkflowMgm.Init("HFSI-Leidenfrost-Angle-3d");

Project name is set to 'HFSI-Leidenfrost-Angle-3d'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\HFSI-Leidenfrost-Angle-3d'.


In [ ]:
static var myDb = BoSSSshell.WorkflowMgm.DefaultDatabase;

## Grid and Boundary/Initial Value configuration

In [7]:
static class Constants {
    // careful order of declaration matters!!!
    // material parameters
    public static double rho_A = 200.0;
    public static double rho_B = 5.0;

    public static double mu_A = 0.3;
    public static double mu_B = 0.005;

    public static double Sigma = 0.1;

    public static double c_A = 400.0;
    public static double c_B = 200.0;

    public static double k_A = 40.0;
    public static double k_B = 1.0;

    public static double hVap = 10000;
    public static double T_sat = 0; // 500K, shifted
    public static double T_wall = T_sat + 5.0;

    public static double alpha_A = k_A / (c_A * rho_A);
    public static double alpha_B = k_B / (c_B * rho_B);
    public static double eps = 1.0 - rho_B / rho_A;

    public static double g = 9.81;
    //public static double g = 9.81 * Math.Sqrt(2.0) / 2.0;

    public static double lambda = 2 * Math.PI * Math.Sqrt((3 * Sigma) / (g * Math.Abs(rho_A - rho_B)));
    public static double L = lambda / 2.0;

    public static double ST = 0.995 * L * 3.0 / 4.0; // 0.01; Solid Thickness
    public static double X0 = 0 * L;
    public static double Y0 = L * (1.0 / 4.0 + 1.0 / 16.0) + ST;
    public static double R = L / 4.0;

    // capillary timestep, for finest res + highest degree, use this, for comparability?! 
    // Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) =>
        0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [16]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 *res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 6 * Constants.L, 6*res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergence(int res, IDatabaseInfo myDb){
        double[] Xnodes = GenericBlas.Linspace(- Constants.L, 11.0 * Constants.L, 12 * res + 1);
        double trsh = 0.05 * Constants.ST;
        //double[] Ynodes = GenericBlas.Linspace(0, Constants.ST - 2 * trsh, res + 1);
        //Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh, 2 * Constants.L, 2 * res + 1));
        double[] Ynodes = GenericBlas.Linspace(0, 2.0 * Constants.L, 2 * res + 1);
        double[] Znodes = GenericBlas.Linspace(- Constants.L, 0.0 * Constants.L, res + 1);


        var grd = Grid3D.Cartesian3DGrid(Xnodes, Ynodes, Znodes);

        //grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(1, "wall_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        grd.EdgeTagNames.Add(4, "freeslip_ZeroGradient");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            else if((Math.Abs(X[2] - Znodes.Last()) < 1e-6) )
                return 4;
            else if((Math.Abs(X[2] - Znodes.First()) < 1e-6) )
                //return 1;
                return 4;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergencePeriodic(int res, IDatabaseInfo myDb){
        //double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(- Constants.L, 6.0 * Constants.L, 7 * res + 1);
        double trsh = 0.05 * Constants.ST;
        double[] Ynodes = GenericBlas.Linspace(0, Constants.ST - 2 * trsh, res + 1);
        Ynodes = Ynodes.Cat(GenericBlas.Linspace(Constants.ST + trsh, 2 * Constants.L, 2 * res + 1));
        //double[] Ynodes = GenericBlas.Linspace(0, 2 * Constants.L, 2 * res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: true);

        grd.EdgeTagNames.Add(1, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[1] - Ynodes.Last()) < 1e-8) )
                return 1;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-8) )
                return 2;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [17]:
public static class BoundaryAndInitialValueFactory {

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine($"         return 1.0 * ((X[0]-{Constants.X0}).Pow2() + (X[1]-{Constants.Y0}).Pow2() + (X[2]).Pow2() - {Constants.R}.Pow2());");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi1(double[] X, double t){");
            //stw.WriteLine($"         return -(X[1] - {Constants.ST});");
            stw.WriteLine($"         return -(X[1] - {Constants.ST});");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }

    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode: GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode: GetPrefixCode());
    }
    static public Formula Get_Phi1() {
        return new Formula("BoundaryAndInitialValues.Phi1", true, AdditionalPrefixCode: GetPrefixCode());
    }
}

## Initialize Controlfiles

In [ ]:
public static HFSI_Control Leidenfrost(int res = 2, int lvl = 2, int p = 2) {
        HFSI_Control C = new HFSI_Control(p);

        // basic database options
        // ======================
        C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
        C.SessionName = "HFSI-Leidenfrost" + "_H" + res + "_AMR" + lvl + "_P" + p + "_Angle_3D_MK5_pressure_Soft";
        C.ProjectName = "HFSI-Leidenfrost";
        C.ProjectDescription = "Leidenfrost on HFSI";
        
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Convtype", "H"));


        // DG degrees
        // ==========
        //C.SetDGdegree(p);

        // Physical Parameters
        // ===================
        C.solveCoupledHeatEquation    = true;
        C.PhysicalParameters.Material = false;

        C.PhysicalParameters.IncludeConvection = true;
        C.ThermalParameters.IncludeConvection  = true;

        C.PhysicalParameters.rho_A = Constants.rho_A;
        C.PhysicalParameters.rho_B = Constants.rho_B;

        C.PhysicalParameters.mu_A = Constants.mu_A;
        C.PhysicalParameters.mu_B = Constants.mu_B;

        C.PhysicalParameters.Sigma = Constants.Sigma;

        C.ThermalParameters.rho_A = Constants.rho_A;
        C.ThermalParameters.rho_B = Constants.rho_B;
        C.ThermalParameters.rho_C = Constants.rho_A;

        C.ThermalParameters.c_A = Constants.c_A;
        C.ThermalParameters.c_B = Constants.c_B;
        C.ThermalParameters.c_C = Constants.c_A;

        C.ThermalParameters.k_A = Constants.k_A;
        C.ThermalParameters.k_B = Constants.k_B;
        C.ThermalParameters.k_C = Constants.k_A * 10.0;

        C.ThermalParameters.hVap  = Constants.hVap;
        C.ThermalParameters.T_sat = Constants.T_sat;

        C.PhysicalParameters.betaS_A = 0.0;
        C.PhysicalParameters.betaS_B = 0.0;

        C.PhysicalParameters.betaL      = 0.0;
        C.PhysicalParameters.sliplength = 0.0;
        C.PhysicalParameters.theta_e    = Math.PI / 2.0; 

        C.Material = new Solid() {
            Density = Constants.rho_A,
            Lame2 = 1e2,
            //Lame2 = 1e10,
            Viscosity = Constants.mu_A
        };

        // grid generation
        // ===============
        bool periodic = false;
        var grd = periodic ? GridFactory.GridConvergencePeriodic(res, BoSSSshell.WorkflowMgm.DefaultDatabase) : GridFactory.GridConvergence(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
        C.SetGrid(grd);
        
        // Initial Values
        // ==============
        C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
        C.AddInitialValue("Phi2", BoundaryAndInitialValueFactory.Get_Phi1());
        C.AddInitialValue("Temperature#A", "X => "+Constants.T_sat, false);
        C.AddInitialValue("Temperature#B", BoundaryAndInitialValueFactory.Get_TempB());
        C.AddInitialValue("Temperature#C", $"X => + {Constants.T_sat} + ({Constants.T_wall}-{Constants.T_sat}) * (1.0- X[1]/{Constants.ST})", false);
        C.AddInitialValue("GravityX#A", new Formula($"X =>  {Constants.g} * Math.Sin(Math.PI / 6)",false));
        C.AddInitialValue("GravityY#A", new Formula($"X => -{Constants.g} * Math.Cos(Math.PI / 6)",false));
        //C.AddInitialValue("GravityX#B", new Formula($"X =>  {Constants.g} * Math.Sin(Math.PI / 6)",false));
        //C.AddInitialValue("GravityY#B", new Formula($"X => -{Constants.g} * Math.Cos(Math.PI / 6)",false));
        //C.AddInitialValue("GravityY#C", new Formula($"X => -{Constants.g}",false));


        // boundary conditions
        // ===================
        if(!periodic) C.AddBoundaryValue("wall_ZeroGradient");
        C.AddBoundaryValue("freeslip_ZeroGradient");
        C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", "X => " + Constants.T_sat, false);
        if(Constants.ST > 0.0) {
            C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#C", "X => " + Constants.T_wall, false);
        } else {
            C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#B", "X => " + Constants.T_wall, false);
        }


        // level set options
        // =================
        C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
        C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
        C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;


        // Timestepping
        // ============
        C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
        C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.BDF2;
        C.dtFixed          = Constants.dt((int)(res * Math.Pow(2, lvl)), p); // Use same timestep for all, better comparability
        C.Endtime          = 5.0;
        C.NoOfTimesteps    = 300;//(int)Math.Ceiling(C.Endtime /C.dtFixed);
        C.saveperiod       = 1;
        C.rollingSaves     = false;
        
        // AMR
        // ============
        int level = lvl;
        C.AdaptiveMeshRefinement = lvl > 0;
        C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 0, maxRefinementLevel = level});
        //C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ levelSet = 1, maxRefinementLevel = level - 1});
        C.AMR_startUpSweeps = lvl;
        //C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRatInnerContactLine { maxRefinementLevel = level, levelSets = new[] { 0, 1 }, FieldWidth = 1 });

        C.DynamicLoadBalancing_On = true;
        C.DynamicLoadBalancing_Period = 20;
        C.DynamicLoadBalancing_RedistributeAtStartup = true;
        C.GridPartType = GridPartType.METIS;

        C.ReInitPeriod = 100;

        // special options
        // ===============
        C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig();
        C.NonLinearSolver.MaxSolverIterations = 25;
        C.NonLinearSolver.MinSolverIterations = 3;
        C.NonLinearSolver.ConvergenceCriterion = 1e-8;
        C.FailOnSolverFail = false;
        //C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MassfluxLogging() { LogPeriod = 1 }); // further logging?

        return C;
}

In [ ]:
if (BoSSS.Application.BoSSSpad.WorkflowMgm.RunWorkflowFromBackup) {
    // in backup mode, the database is a pre-computed snapshot -- there is nothing to submit
    // or wait for, and touching the batch processor here would try to correlate against a job
    // whose deployment directory may no longer exist.
    Console.WriteLine("Running from backup database; skipping job submission.");
} else {
    var C_CTRLFILE = Leidenfrost(1,4,2);//Create control file.
    var job = C_CTRLFILE.CreateJob();
    job.NumberOfMPIProcs = 8;
    job.NumberOfThreads = 4;
    job.Activate(myBatch);
    //job.ShowOutput();

    BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(864000);

    int count = BoSSSshell.WorkflowMgm.Sessions.Count();
    int success = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination).Count();

    if(count != success) {
        throw new ApplicationException("Not all simulations calculated or finished successful!");
    }
}